# Util embedding experiments

In [1]:
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

HERE = Path.cwd()
sys.path.insert(0, str(HERE))

import datasets
import embed

/home/matthew/Projects/2025_moral_feature_modeling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load stimuli
`SUBSETS` maps each subset label -> {"texts": list[str], "y": float array} for the directions we fit.

In [2]:
SUBSETS = {}  # label -> {"texts": list[str], "y": np.ndarray(float)}
# Loaders first, then SUBSETS in presentation order: value axes (higher = better)
# followed by the severity axes (higher = more).
ethics = datasets.load_ethics()
exp1, exp2 = datasets.load_franken()
dillion = datasets.load_dillion()
aita = datasets.load_aita()
gbd = datasets.load_gbd()
hr = datasets.load_holmes_rahe()

# --- value axes: higher = better ---
cm = ethics["commonsense"]["train"]
cm_short = cm[cm["is_short"]]
SUBSETS["ETHICS-cm"] = {"texts": cm_short["input"].tolist(), "y": (1 - cm_short["label"]).to_numpy(float)}  # flip: 1 = acceptable (good), 0 = wrong

a, b = ethics["utilitarianism"]["train"]["more_pleasant"], ethics["utilitarianism"]["train"]["less_pleasant"]
SUBSETS["ETHICS-util"] = {"texts": list(a) + list(b), "y": np.r_[np.ones(len(a)), np.zeros(len(b))]}

SUBSETS["dillion"] = {"texts": dillion["Situation"].tolist(), "y": dillion["Human rating"].to_numpy(float)}
SUBSETS["AITA-utility"] = {"texts": aita["outcome"].tolist(), "y": aita["utility"].to_numpy(float)}
SUBSETS["franken-valence"] = {"texts": exp1["target"].tolist(), "y": exp1["avg_likert_rating"].to_numpy(float)}
SUBSETS["franken-permissibility"] = {"texts": exp2["text"].tolist(), "y": exp2["avg_permissibility_rating"].to_numpy(float)}
SUBSETS["franken-severe_vs_mild"] = {"texts": exp1["target"].tolist(), "y": (exp1["strength"] == "severe").to_numpy(float)}

# --- severity axes: higher = more ---
SUBSETS["GBD"] = {"texts": gbd["lay_description"].tolist(), "y": gbd["weight"].to_numpy(float)}
SUBSETS["Holmes-Rahe"] = {"texts": hr["event"].tolist(), "y": hr["lcu"].to_numpy(float)}

for label, sub in SUBSETS.items():
    print(f"{label:24s} n={len(sub['texts']):5d}")

ETHICS-cm                n= 6661
ETHICS-util              n=27476
dillion                  n=  464
AITA-utility             n=   59
franken-valence          n=   80
franken-permissibility   n=   80
franken-severe_vs_mild   n=   80
GBD                      n=  233
Holmes-Rahe              n=   43


## 2. Get embeddings
Switch `MODEL` to any name in `embed.OPENAI_MODELS` or `embed.QWEN_MODELS` to change model.

In [3]:
import os
MODEL = "Qwen/Qwen3-Embedding-4B"  # or any of embed.OPENAI_MODELS / embed.QWEN_MODELS
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")  # only needed if MODEL is an OpenAI model

embeddings = {}
for label, sub in SUBSETS.items():
    emb = embed.embed_dataset(sub["texts"], dataset=label, model=MODEL, api_key=OPENAI_API_KEY)
    X = np.array([emb[str(t)] for t in sub["texts"]])
    embeddings[label] = X
    print(f"{label:24s} {X.shape}")

ETHICS-cm                (6661, 2560)


ETHICS-util              (27476, 2560)
dillion                  (464, 2560)
AITA-utility             (59, 2560)
franken-valence          (80, 2560)
franken-permissibility   (80, 2560)
franken-severe_vs_mild   (80, 2560)
GBD                      (233, 2560)
Holmes-Rahe              (43, 2560)


## 3. Fit directions
Two simple methods to compare: ridge regression of the embeddings on `y`, and a plain
mean-difference between the high and low halves (split at the median for continuous `y`).

In [4]:
def ridge_dir(X, y, alpha=50.0):
    Xc = X - X.mean(0)
    w = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(Xc.shape[1]), Xc.T @ (y - y.mean()))
    return w / (np.linalg.norm(w) + 1e-9)

def meandiff_dir(X, y):
    classes = np.unique(y)
    if len(classes) == 2:            # binary: split by class membership, not by a threshold
        hi, lo = X[y == classes[1]].mean(0), X[y == classes[0]].mean(0)
    else:                            # continuous: split at the median
        thresh = np.median(y)
        hi, lo = X[y >= thresh].mean(0), X[y < thresh].mean(0)
    d = hi - lo
    return d / (np.linalg.norm(d) + 1e-9)

METHOD = "ridge"  # or "meandiff"

directions = {}
for label, sub in SUBSETS.items():
    X = embeddings[label]
    d = ridge_dir(X, sub["y"]) if METHOD == "ridge" else meandiff_dir(X, sub["y"])
    directions[label] = d

## 4. Evaluating directions within-dataset

### 4a. Correlation - continuous `y`
For datasets with continuous labels, we can evaluate how well a single direction captures the labels by computing the correlation between projections onto the direction and the continuous labels `y`.

In [5]:
CONTINUOUS = [k for k, s in SUBSETS.items() if len(np.unique(s["y"])) > 2]

labels_c, rs = [], []
for label in CONTINUOUS:
    proj = embeddings[label] @ directions[label]
    rs.append(np.corrcoef(proj, SUBSETS[label]["y"])[0, 1])
    labels_c.append(label)

fig = go.Figure(go.Bar(x=labels_c, y=rs, text=rs, texttemplate="%{text:.2f}", textposition="outside"))
fig.update_layout(title="Correlation between y and projection",
                  yaxis_title="Pearson r", xaxis_tickangle=-60, width=900, height=500)
fig.show()

### 4b. Binary classification - two-class `y`
For subsets whose `y` is a genuine two-class label, correlation is not the natural summary.
Use AUC: the probability that a randomly chosen positive item projects higher than a
randomly chosen negative one. 0.5 = chance, 1.0 = perfect separation.

In [6]:
BINARY = [k for k, s in SUBSETS.items()
          if len(np.unique(s["y"])) == 2 and k != "ETHICS-util"]

def auc(scores, y):
    """Rank-based AUC; ties get average rank."""
    order = np.argsort(scores)
    ranks = np.empty(len(scores), float)
    ranks[order] = np.arange(1, len(scores) + 1)
    pos, neg = y == 1, y == 0
    n1, n0 = pos.sum(), neg.sum()
    return (ranks[pos].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

labels_b, aucs = [], []
for label in BINARY:
    proj = embeddings[label] @ directions[label]
    aucs.append(auc(proj, SUBSETS[label]["y"]))
    labels_b.append(label)

fig = go.Figure(go.Bar(x=labels_b, y=aucs, text=aucs, texttemplate="%{text:.3f}", textposition="outside"))
fig.add_hline(y=0.5, line_dash="dash", line_color="grey", annotation_text="chance")
fig.update_layout(title="In-sample AUC", yaxis_title="AUC",
                  yaxis_range=[0.4, 1.02], xaxis_tickangle=-60, width=700, height=500)
fig.show()

### 4c. Pairwise binary classification - ranked pairs
For datasets where the labels are pairwise binary (ETHICS-util, which is 'more pleasant' vs 'less pleasant'), its native metric is **pairwise accuracy**: within each pair, does the more-pleasant scenario project higher?

In [7]:
label = "ETHICS-util"
proj = embeddings[label] @ directions[label]
n_pairs = len(proj) // 2
hi, lo = proj[:n_pairs], proj[n_pairs:]          # more_pleasant, less_pleasant

pairwise_acc = float((hi > lo).mean())
pooled_r = float(np.corrcoef(proj, SUBSETS[label]["y"])[0, 1])
print(f"{label}: {n_pairs:,} pairs")
print(f"  pairwise accuracy = {pairwise_acc:.3f}   (chance = 0.5)")
fig = go.Figure(go.Bar(x=["pairwise accuracy",], y=[pairwise_acc,],
                       text=[pairwise_acc,], texttemplate="%{text:.3f}", textposition="outside"))
fig.add_hline(y=0.5, line_dash="dash", line_color="grey", annotation_text="chance")
fig.update_layout(title=f"{label}: native pairwise metric vs pooled correlation",
                  yaxis_range=[0, 1.05], width=300, height=500)
fig.show()

ETHICS-util: 13,738 pairs
  pairwise accuracy = 0.865   (chance = 0.5)


## 5. Cosine similarity between directions (Cross-Dataset)
What we really want is a direction which generalized *across* datasets. One way is to look at the geometry of the directions -- how similar is the direction learned for one dataset (e.g. franken-valence) to the direction learned for another dataset (e.g. AITA-utility). To do this we use cosine similarity.

In [8]:
labels = list(directions.keys())
D = np.array([directions[l] for l in labels])
C = D @ D.T

fig = px.imshow(C, x=labels, y=labels, color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.update_yaxes(scaleanchor="x", constrain="domain")
fig.update_xaxes(constrain="domain")
fig.update_layout(title="Direction x direction cosine similarity", width=820, height=760)
fig.show()

## 6. Correlations between labels and projections (Cross-Dataset)
Like section 5, but cross-dataset: for every (labels_i, direction_j) pair, project
subset i's embeddings onto subset j's fitted direction and correlate with subset i's
own `y`. Row = whose data/labels, column = whose direction. The diagonal reproduces
section 4's numbers; off-diagonal cells show how well one dataset's direction
generalizes to another's labels (e.g. franken-valence data projected onto the
nie-causal_role direction, correlated with the franken-valence labels).

In [9]:
Cross = np.zeros((len(labels), len(labels)))
for i, li in enumerate(labels):
    Xi, yi = embeddings[li], SUBSETS[li]["y"]
    for j, lj in enumerate(labels):
        proj = Xi @ directions[lj]
        Cross[i, j] = np.corrcoef(proj, yi)[0, 1]

fig = px.imshow(Cross, x=labels, y=labels, color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.update_yaxes(scaleanchor="x", constrain="domain")
fig.update_xaxes(constrain="domain")
fig.update_layout(title="Correlation between y and projection, cross-dataset (row = data, column = direction)",
                   width=820, height=760)
fig.update_xaxes(title="direction")
fig.update_yaxes(title="data / labels")
fig.show()